# Evaluation Notebook

This notebook can be used for evaluating offline trained CHUNGUS models.

The functionality of this notebook differs slightly from that of test.py, as that evaluation is performed on extracted embeddings (rather than mimicing the whole flow of image->DINOv2->FeatUp->traversability predictor, including any resizing applied that would be used during robot deployment). Thus, the results should be very similar but may not always be perfectly matching. This notebook should be considered the preferred way to compute metrics, as it considers the entire pipeline of prediction.

In [ ]:
import sys
sys.path.append('./../')

import torch
from utils import *
from pathlib import Path
from chungus import DINOv2FeatUp

import warnings
warnings.filterwarnings('ignore')

### Specify data

In [ ]:
image_folder = Path('../../data/outdoor/images/') # Folder for the images to use
csv_path = Path('../../data/outdoor/val_outdoor_combined.csv') # CSV folder with annotations
resolution = (240, 424) # Resolution of images

### Specify network settings

In [ ]:
device = torch.device('cuda') # device for inference (must be cuda device for FeatUp)
core_model_resolution = (224,224) # Resolution of model inference (rescaled back to original resolution automatically)
model_file = './../experiments/outdoor_dinov2_224x224/best_model.pth' # PyTorch model file

In [ ]:
# Create network
network = DINOv2FeatUp(core_size=core_model_resolution, output_size=resolution)
network.load_model(model_file)
network.eval()
network.to(device)
print("Model loaded")

### Evaluation

In [ ]:
# HDR thresholds to evaluate on
thresholds = [
    0.1,
    0.25,
    0.5
]

# Evaluate
evaluation_results = evaluate(csv_path, image_folder, network, thresholds, device)

# Print the results of the evaluation
print("\tEvaluation Results")
for threshold in thresholds:
    print("HDR @ {:.2f} = {:.3f}".format(threshold, evaluation_results[threshold]))